# Task 1 - Part 3 Model Training Pipeline

## recommender_fm.py
Implementation of an FM (Factorization Machine) for recommending groceries. This is a supervised learning algorithm suited for sparse, high-dimensional (possess a large number of features) datasets, suiting it for recommendation systems. It captures interactions between input features, even between features that haven't co-occurred in the training set, by representing each feature as a learned latent embedding. By calculating the inner product of these embeddings, the model can predict the strength of pairwise relationships, even if the combinations are rare.
The FM trains on categorical interactions:
- `user_id` - representing the shopper
- `product_id` - representing the product
- Latent embeddings - IDs are mapped into a shared 32-dimensional space
A baseline "popularity" is recorded for each user and product and then the inner product of the user and product embeddings are calculated.

In [ ]:
import os
import time
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from dataclasses import dataclass, field
from typing import List
from safetensors.torch import save_file

import run_utils
from config import get_run_config, RecommenderMetrics


@dataclass
class FMConfig:
    run: str = "F1"
    embed_dim: int = 32
    learning_rate: float = 0.001
    batch_size: int = 4096
    epochs: int = 1
    sample_frac: float = 0.1
    model_type: str = "recommender_fm"


class FactorizationMachine(nn.Module):
    def __init__(self, field_dims, embed_dim):
        super().__init__()
        self.offsets = np.array((0, *np.cumsum(field_dims)[:-1]))

        self.linear = nn.Embedding(sum(field_dims), 1)
        self.embedding = nn.Embedding(sum(field_dims), embed_dim)
        self.bias = nn.Parameter(torch.zeros(1))

        nn.init.xavier_uniform_(self.embedding.weight)

    def forward(self, x):
        x = x + x.new_tensor(self.offsets).unsqueeze(0)

        linear = torch.sum(self.linear(x), dim=1) + self.bias

        emb = self.embedding(x)
        square_of_sum = torch.sum(emb, dim=1) ** 2
        sum_of_square = torch.sum(emb ** 2, dim=1)

        interaction = 0.5 * \
            torch.sum(square_of_sum - sum_of_square, dim=1, keepdim=True)

        return torch.sigmoid(linear + interaction).squeeze()


def load_data(filepath='insta_clean_data.parquet', sample_frac=1.0):
    print(f"--- Loading data (path: {filepath}, frac: {sample_frac})")
    df = pd.read_parquet(filepath)

    if sample_frac < 1.0:
        df = df.sample(frac=sample_frac, random_state=42)

    df['days_since_last_order'] = df['days_since_last_order'].fillna(0)

    # Split
    train_df = df[df['eval_set'] == 'prior'].copy()
    test_df = df[df['eval_set'] == 'train'].copy()

    # Time-aware validation split
    train_df = train_df.sort_values(['user_id', 'order_number'])
    val_idx = train_df.groupby('user_id')['order_number'].transform(
        lambda x: x >= x.max() - 1
    )

    val_df = train_df[val_idx].copy()
    train_df = train_df[~val_idx].copy()

    cat_features = ['user_id', 'product_id']

    # Encode IDs to match NCF structure for Top-K logic
    user_map = {u: i for i, u in enumerate(df['user_id'].unique())}
    item_map = {p: i for i, p in enumerate(df['product_id'].unique())}
    product_name_map = df[['product_id', 'product_name']].drop_duplicates().set_index('product_id')['product_name'].to_dict()
    
    for d in [train_df, val_df, test_df]:
        d['user_id'] = d['user_id'].map(user_map)
        d['product_id'] = d['product_id'].map(item_map)

    field_dims = [len(user_map), len(item_map)]

    # Extract arrays
    X_train = train_df[cat_features].values
    y_train = train_df['is_reorder'].values

    X_val = val_df[cat_features].values
    y_val = val_df['is_reorder'].values

    X_test = test_df[cat_features].values
    y_test = test_df['is_reorder'].values

    return (X_train, y_train, X_val, y_val, X_test, y_test, 
            field_dims, len(user_map), len(item_map), 
            test_df, user_map, item_map, product_name_map, train_df)


def calculate_precision_recall_at_k(model, u_test, i_test, y_test, num_items, device, k=10):
    model.eval()
    user_test_data = pd.DataFrame({
        'user_id': u_test,
        'product_id': i_test,
        'label': y_test
    })

    positive_users = user_test_data[user_test_data['label'] == 1]['user_id'].unique()
    if len(positive_users) == 0:
        return 0.0, 0.0

    precisions = []
    recalls = []

    sampled_users = np.random.choice(positive_users, min(100, len(positive_users)), replace=False)

    for u_idx in sampled_users:
        actual_relevant = set(user_test_data[(user_test_data['user_id'] == u_idx) & (
            user_test_data['label'] == 1)]['product_id'].values)

        x_vec = torch.stack([
            torch.full((num_items,), u_idx, dtype=torch.long),
            torch.arange(num_items, dtype=torch.long)
        ], dim=1).to(device)

        with torch.no_grad():
            scores = model(x_vec).cpu().numpy()

        top_k_idx = np.argsort(scores)[-k:][::-1]
        top_k_items = set(top_k_idx)

        hits = len(actual_relevant.intersection(top_k_items))
        precisions.append(hits / k)
        recalls.append(hits / len(actual_relevant))

    return np.mean(precisions), np.mean(recalls)


def predict_random_items(model, user_idx, num_items, device, k=5):
    model.eval()
    item_indices = np.random.choice(num_items, size=k, replace=False)

    x_vec = torch.stack([
        torch.full((k,), user_idx, dtype=torch.long),
        torch.tensor(item_indices, dtype=torch.long)
    ], dim=1).to(device)

    with torch.no_grad():
        scores = model(x_vec).cpu().numpy()

    return item_indices, scores


def train_fm(run_id="D1", output_dir="recommender_results"):
    config = FMConfig(run=run_id)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    (X_train, y_train, X_val, y_val, X_test, y_test, 
     field_dims, num_users, num_items, 
     test_df, user_map, item_map, product_name_map, train_df) = load_data(sample_frac=config.sample_frac)

    train_loader = DataLoader(
        TensorDataset(torch.tensor(X_train, dtype=torch.long),
                      torch.tensor(y_train, dtype=torch.float32)),
        batch_size=config.batch_size, shuffle=True
    )
    val_loader = DataLoader(
        TensorDataset(torch.tensor(X_val, dtype=torch.long),
                      torch.tensor(y_val, dtype=torch.float32)),
        batch_size=config.batch_size, shuffle=False
    )

    model = FactorizationMachine(field_dims, config.embed_dim).to(device)
    optimizer = optim.Adam(model.parameters(), lr=config.learning_rate)
    criterion = nn.BCELoss()

    metrics = RecommenderMetrics(run=f"{config.run}_metrics")
    start_time = time.time()

    print(f"--- Training FM Run {config.run}")
    for epoch in range(config.epochs):
        model.train()
        total_loss = 0
        num_batches = len(train_loader)
        for i, (Xb, yb) in enumerate(train_loader):
            Xb, yb = Xb.to(device), yb.to(device)
            optimizer.zero_grad()
            preds = model(Xb)
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        model.eval()
        val_acc = 0
        total_test = 0
        with torch.no_grad():
            for Xb, yb in val_loader:
                Xb, yb = Xb.to(device), yb.to(device)
                preds = model(Xb)
                val_acc += ((preds > 0.5) == yb).sum().item()
                total_test += yb.size(0)

        final_val_acc = val_acc / total_test
        avg_train_loss = total_loss / num_batches
        metrics.epochs.append(epoch + 1)
        metrics.train_loss.append(avg_train_loss)
        metrics.test_accuracy.append(final_val_acc)
        print(f"Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val Acc: {final_val_acc:.4f}")

    metrics.elapsed_time_min = (time.time() - start_time) / 60
    metrics.final_accuracy = metrics.test_accuracy[-1]

    os.makedirs(output_dir, exist_ok=True)
    save_file(model.state_dict(), os.path.join(output_dir, f'{config.run}.safetensors'))

    print("--- Calculating Precision@K and Recall@K...")
    u_test = X_test[:, 0]
    i_test = X_test[:, 1]
    p_at_k, r_at_k = calculate_precision_recall_at_k(
        model, u_test, i_test, y_test, num_items, device, k=10
    )
    metrics.precision_at_k = p_at_k
    metrics.recall_at_k = r_at_k
    print(f"Precision@10: {p_at_k:.4f} | Recall@10: {r_at_k:.4f}")

    run_utils.save_run_data(config, metrics, output_dir)

    # TOP-K & HISTORY SAMPLE FOR DASHBOARD
    model.eval()
    with torch.no_grad():
        unique_test_users = test_df['user_id'].unique()
        sample_user_ids = [np.random.choice(unique_test_users)]

        inv_user_map = {i: u for u, i in user_map.items()}
        inv_item_map = {i: p for p, i in item_map.items()}

        u_idx = sample_user_ids[0]
        items, scores = predict_random_items(model, u_idx, num_items, device, k=5)
        for i, s in zip(items, scores):
            pid = inv_item_map[i]
            name = product_name_map.get(pid, str(pid))
            print(f"{name} → {s:.4f}")

        top_k_list = []
        history_list = []
        k = 10

        for u_idx in sample_user_ids:
            # 1. History
            u_hist = train_df[train_df['user_id'] == u_idx].sort_values('order_number', ascending=False).head(5)
            for _, row in u_hist.iterrows():
                history_list.append({
                    'user_id_orig': str(inv_user_map[u_idx]),
                    'product_name': product_name_map.get(inv_item_map[row['product_id']], "Unknown")
                })

            # 2. Top-K Recommendations
            x_vec = torch.stack([
                torch.full((num_items,), u_idx, dtype=torch.long),
                torch.arange(num_items, dtype=torch.long)
            ], dim=1).to(device)
            scores = model(x_vec).cpu().numpy()

            top_indices = np.argsort(scores)[-k:][::-1]
            for rank, i_idx in enumerate(top_indices):
                top_k_list.append({
                    'user_id': u_idx,
                    'user_id_orig': str(inv_user_map[u_idx]),
                    'product_id': i_idx,
                    'product_name': product_name_map.get(inv_item_map[i_idx], "Unknown"),
                    'score': scores[i_idx],
                    'rank': rank + 1
                })

        top_k_df = pd.DataFrame(top_k_list)
        history_df = pd.DataFrame(history_list)

    # DASHBOARD
    run_utils.plot_ncf_dashboard(config, metrics, top_k_df, history_df, output_dir)

    print(f"FM training complete: {config.run}")

if __name__ == "__main__":
    train_fm("F1")


## recommender_ncf.py
Neural Collaborative Filtering is a supervised learning algorithm using neural networks (i.e. a multi-layer perceptron) for learning complex, non-linear user-item interaction functions as opposed to linear matrix factorization in order to capture the kind of user-item relationships overlooked by typical linear models.
This model is trained on user-item interaction data, specifically focused on the re-purchase behaviour reflected in the "is_reorder" column. Binary labels (0 for non-reorder, 1 for reorder) are used to learn from user actions.
User and item embeddings are joined into a single long vector, passed through multiple hidden layers to learn "high-order" interactions (patterns where a user's preference for a product depends on complex, non-linear combinations of previous habits) and passed through a sigmoid function to produce a score between 0 and 1 reflecting the probability of interest.

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from dataclasses import dataclass
from safetensors.torch import save_file

import run_utils
from config import RecommenderMetrics


# CONFIG
@dataclass
class NCFConfig:
    run: str = "E1"
    embed_dim: int = 32
    hidden_dims: list = None
    learning_rate: float = 0.001
    batch_size: int = 4096
    epochs: int = 1
    sample_frac: float = 0.1
    model_type: str = "recommender_ncf"

    def __post_init__(self):
        if self.hidden_dims is None:
            self.hidden_dims = [64, 32]


# MODEL
class NCF(nn.Module):
    def __init__(self, num_users, num_items, embed_dim, hidden_dims):
        super().__init__()

        self.user_embed = nn.Embedding(num_users, embed_dim)
        self.item_embed = nn.Embedding(num_items, embed_dim)

        layers = []
        input_dim = embed_dim * 2

        for h in hidden_dims:
            layers.append(nn.Linear(input_dim, h))
            layers.append(nn.ReLU())
            input_dim = h

        layers.append(nn.Linear(input_dim, 1))

        self.mlp = nn.Sequential(*layers)

    def forward(self, user, item):
        u = self.user_embed(user)
        i = self.item_embed(item)

        x = torch.cat([u, i], dim=1)
        return torch.sigmoid(self.mlp(x)).view(-1)


# DATA LOADER
def load_data(filepath='insta_clean_data.parquet', sample_frac=1.0):
    print(f"--- Loading data (NCF): {filepath}, frac={sample_frac}")
    df = pd.read_parquet(filepath)

    if sample_frac < 1.0:
        df = df.sample(frac=sample_frac, random_state=42)

    df['days_since_last_order'] = df['days_since_last_order'].fillna(0)

    # Split
    train_df = df[df['eval_set'] == 'prior'].copy()
    test_df = df[df['eval_set'] == 'train'].copy()

    # Validation split
    train_df = train_df.sort_values(['user_id', 'order_number'])
    val_idx = train_df.groupby('user_id')['order_number'].transform(
        lambda x: x >= x.max() - 1
    )

    val_df = train_df[val_idx].copy()
    train_df = train_df[~val_idx].copy()

    # Encode IDs
    user_map = {u: i for i, u in enumerate(df['user_id'].unique())}
    item_map = {p: i for i, p in enumerate(df['product_id'].unique())}
    product_name_map = df[['product_id', 'product_name']].drop_duplicates(
    ).set_index('product_id')['product_name'].to_dict()

    for d in [train_df, val_df, test_df]:
        d['user_id'] = d['user_id'].map(user_map)
        d['product_id'] = d['product_id'].map(item_map)

    def to_xy(data):
        X_user = data['user_id'].values
        X_item = data['product_id'].values
        y = data['is_reorder'].values
        return X_user, X_item, y

    return (
        *to_xy(train_df),
        *to_xy(val_df),
        *to_xy(test_df),
        len(user_map),
        len(item_map),
        test_df,
        user_map,
        item_map,
        product_name_map,
        train_df
    )


# METRICS
def calculate_precision_recall_at_k(model, u_test, i_test, y_test, num_items, device, k=10):
    model.eval()
    user_test_data = pd.DataFrame({
        'user_id': u_test,
        'product_id': i_test,
        'label': y_test
    })

    # We only care about users who have at least one positive label in test set
    positive_users = user_test_data[user_test_data['label']
                                    == 1]['user_id'].unique()
    if len(positive_users) == 0:
        return 0.0, 0.0

    precisions = []
    recalls = []

    # For efficiency, we sample some users if the test set is large
    sampled_users = np.random.choice(positive_users, min(
        100, len(positive_users)), replace=False)

    for u_idx in sampled_users:
        # Get ground truth items (relevant items)
        actual_relevant = set(user_test_data[(user_test_data['user_id'] == u_idx) & (
            user_test_data['label'] == 1)]['product_id'].values)

        # Predict for all products
        user_tensor = torch.full(
            (num_items,), u_idx, dtype=torch.long).to(device)
        item_tensor = torch.arange(num_items, dtype=torch.long).to(device)

        with torch.no_grad():
            scores = model(user_tensor, item_tensor).cpu().numpy()

        top_k_idx = np.argsort(scores)[-k:][::-1]
        top_k_items = set(top_k_idx)

        hits = len(actual_relevant.intersection(top_k_items))
        precisions.append(hits / k)
        recalls.append(hits / len(actual_relevant))

    return np.mean(precisions), np.mean(recalls)


def predict_random_items(model, user_idx, num_items, device, k=5):
    model.eval()

    # pick random items
    item_indices = np.random.choice(num_items, size=k, replace=False)

    user_tensor = torch.full((k,), user_idx, dtype=torch.long).to(device)
    item_tensor = torch.tensor(item_indices, dtype=torch.long).to(device)

    with torch.no_grad():
        scores = model(user_tensor, item_tensor).cpu().numpy()

    return item_indices, scores

# TRAINING


def train_ncf(run_id="D2", output_dir="recommender_results"):
    config = NCFConfig(run=run_id)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    (
        u_train, i_train, y_train,
        u_val, i_val, y_val,
        u_test, i_test, y_test,
        num_users, num_items,
        test_df,
        user_map,
        item_map,
        product_name_map,
        train_df
    ) = load_data(sample_frac=config.sample_frac)

    train_loader = DataLoader(
        TensorDataset(
            torch.tensor(u_train, dtype=torch.long),
            torch.tensor(i_train, dtype=torch.long),
            torch.tensor(y_train, dtype=torch.float32)
        ),
        batch_size=config.batch_size,
        shuffle=True
    )

    val_loader = DataLoader(
        TensorDataset(
            torch.tensor(u_val, dtype=torch.long),
            torch.tensor(i_val, dtype=torch.long),
            torch.tensor(y_val, dtype=torch.float32)
        ),
        batch_size=config.batch_size,
        shuffle=False
    )

    model = NCF(num_users, num_items,
                config.embed_dim, config.hidden_dims).to(device)

    optimizer = optim.Adam(model.parameters(), lr=config.learning_rate)
    criterion = nn.BCELoss()

    metrics = RecommenderMetrics(run=config.run)

    print(f"--- Training NCF {config.run}")
    start_time = time.time()

    for epoch in range(config.epochs):
        model.train()
        total_loss = 0

        for u, i, y in train_loader:
            u, i, y = u.to(device), i.to(device), y.to(device)

            optimizer.zero_grad()
            preds = model(u, i)
            loss = criterion(preds, y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        # Validation
        model.eval()
        correct = 0
        total = 0

        with torch.no_grad():
            for u, i, y in val_loader:
                u, i, y = u.to(device), i.to(device), y.to(device)
                preds = model(u, i)
                correct += ((preds > 0.5) == y).sum().item()
                total += y.size(0)

        acc = correct / total
        avg_loss = total_loss / len(train_loader)

        metrics.epochs.append(epoch + 1)
        metrics.train_loss.append(avg_loss)
        metrics.test_accuracy.append(acc)

        print(f"Epoch {epoch+1} | Loss: {avg_loss:.4f} | Val Acc: {acc:.4f}")

    metrics.final_accuracy = metrics.test_accuracy[-1]
    metrics.elapsed_time_min = (time.time() - start_time) / 60

    # Save model
    os.makedirs(output_dir, exist_ok=True)
    save_file(model.state_dict(),
              os.path.join(output_dir, f"{config.run}.safetensors"))

    # Calculate Precision@K and Recall@K
    print("--- Calculating Precision@K and Recall@K...")
    p_at_k, r_at_k = calculate_precision_recall_at_k(
        model, u_test, i_test, y_test, num_items, device, k=10
    )
    metrics.precision_at_k = p_at_k
    metrics.recall_at_k = r_at_k
    print(f"Precision@10: {p_at_k:.4f} | Recall@10: {r_at_k:.4f}")

    run_utils.save_run_data(config, metrics, output_dir)

    # TOP-K & HISTORY SAMPLE FOR DASHBOARD
    model.eval()
    with torch.no_grad():
        # Pick 1 user from test set
        # Pick 3 users from test set
        unique_test_users = test_df['user_id'].unique()

        # sample_user_ids = np.random.choice(unique_test_users, min(3, len(unique_test_users)), replace=False)
        sample_user_ids = [np.random.choice(unique_test_users)]

        # new
        # create mappings
        inv_user_map = {i: u for u, i in user_map.items()}
        inv_item_map = {i: p for p, i in item_map.items()}

        u_idx = sample_user_ids[0]  # your selected user
        items, scores = predict_random_items(
            model, u_idx, num_items, device, k=5)

        for i, s in zip(items, scores):
            pid = inv_item_map[i]
            name = product_name_map.get(pid, str(pid))
            print(f"{name} → {s:.4f}")
        # new

        top_k_list = []
        history_list = []
        k = 10

        inv_user_map = {i: u for u, i in user_map.items()}
        inv_item_map = {i: p for p, i in item_map.items()}

        for u_idx in sample_user_ids:
            # 1. History
            # Extract last 5 from train_df
            u_hist = train_df[train_df['user_id'] == u_idx].sort_values(
                'order_number', ascending=False).head(5)
            for _, row in u_hist.iterrows():
                history_list.append({
                    'user_id_orig': str(inv_user_map[u_idx]),
                    'product_name': product_name_map.get(inv_item_map[row['product_id']], "Unknown")
                })

            # 2. Top-K Recommendations
            user_tensor = torch.full(
                (num_items,), u_idx, dtype=torch.long).to(device)
            item_tensor = torch.arange(num_items, dtype=torch.long).to(device)
            scores = model(user_tensor, item_tensor).cpu().numpy()

            top_indices = np.argsort(scores)[-k:][::-1]
            for rank, i_idx in enumerate(top_indices):
                top_k_list.append({
                    'user_id': u_idx,
                    'user_id_orig': str(inv_user_map[u_idx]),
                    'product_id': i_idx,
                    'product_name': product_name_map.get(inv_item_map[i_idx], "Unknown"),
                    'score': scores[i_idx],
                    'rank': rank + 1
                })

        top_k_df = pd.DataFrame(top_k_list)
        history_df = pd.DataFrame(history_list)

    # DASHBOARD
    run_utils.plot_ncf_dashboard(
        config, metrics, top_k_df, history_df, output_dir)

    print(f"NCF training complete: {config.run}")


if __name__ == "__main__":
    train_ncf("E1")


## xgboost_forecaster.py
Extreme Gradient Boosting is a highly efficient implementation of gradient-boosted decison trees. An ensemble of weak learners, mainly shallow decision trees, is gathered and built sequentially, with each new tree being designed to correct the errors the previous tree.
This model is trained on groceries_dataset.csv, a dataset containing user IDs, dates of purchase and a product description.
- The target variable is the "demand" - the amount of grocery items sold within a specific frequency (i.e. daily or monthly)
- `reindex()` is used to fill empty slots with no sales
- The date column is used to build a temporal profile, creating the `dayofweek`, `month`, `dayofmonth` and `is_weekend`
- `lag` is created for previous demand values and `rolling` features are made to detect "momentum" - the change in demand

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error
import os
import time

import run_utils
from config import get_run_config, ForecastingMetrics


def prepare_data_for_xgb(filepath='groceries_dataset.csv', freq='D'):
    """
    Loads and prepares the dataset specifically for XGBoost.
    Unlike LSTM/SARIMAX, XGBoost requires explicitly engineered tabular features 
    (lags, rolling stats, date parts) instead of raw sequential data.
    """
    if not os.path.exists(filepath):
        print(f"Error: {filepath} not found.")
        return None, None, None, None

    # 1. Load and aggregate to true chronological series
    df = pd.read_csv(filepath)
    df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y')
    df = df.sort_values(by='Date')

    # Handle aggregation based on frequency config
    if freq in ['W', 'ME', 'M']:
        agg = df.groupby(pd.Grouper(key='Date', freq=freq)
                         ).size().reset_index(name='demand')
    else:
        agg = df.groupby('Date').size().reset_index(name='demand')

    agg.set_index('Date', inplace=True)

    full_idx = pd.date_range(start=agg.index.min(),
                             end=agg.index.max(), freq=freq)
    agg = agg.reindex(full_idx, fill_value=0)

    df_ts = agg.copy()

    # 2. Feature Engineering
    # Date parts
    df_ts['dayofweek'] = df_ts.index.dayofweek
    df_ts['month'] = df_ts.index.month
    df_ts['dayofmonth'] = df_ts.index.day
    df_ts['is_weekend'] = df_ts['dayofweek'].isin([5, 6]).astype(int)

    # Dynamically adjust lags and rolling windows based on frequency
    if freq in ['ME', 'M']:
        lags = [1, 2, 3, 6, 12]  # Months
        window_short, window_long = 3, 6
    else:
        lags = [1, 2, 3, 4, 5, 6, 7, 14, 21, 28]  # Days
        window_short, window_long = 7, 14

    for lag in lags:
        df_ts[f'lag_{lag}'] = df_ts['demand'].shift(lag)

    # Rolling features (moving averages and volatility)
    df_ts[f'rolling_mean_{window_short}'] = df_ts['demand'].shift(
        1).rolling(window=window_short).mean()
    df_ts[f'rolling_std_{window_short}'] = df_ts['demand'].shift(
        1).rolling(window=window_short).std()
    df_ts[f'rolling_mean_{window_long}'] = df_ts['demand'].shift(
        1).rolling(window=window_long).mean()

    # Drop rows which now contain NaNs due to the longest lag shift
    df_ts.dropna(inplace=True)

    return df_ts, lags, window_short, window_long


def run_xgboost_forecaster(run_id="C3", output_dir="forecasting_results"):
    print(f"\n--- Starting Standalone XGBoost Forecasting: {run_id} ---")
    start_time = time.time()

    run_config = get_run_config(run_id)
    if not run_config:
        print(f"Run config {run_id} not found.")
        return

    df, lags, window_short, window_long = prepare_data_for_xgb(
        freq=run_config.freq)
    if df is None:
        return

    # Split temporally (80% train, 20% test)
    train_size = int(len(df) * 0.8)
    train_df = df.iloc[:train_size]
    test_df = df.iloc[train_size:]

    features = [col for col in df.columns if col != 'demand']
    target = 'demand'

    X_train, y_train = train_df[features], train_df[target]
    X_test, y_test = test_df[features], test_df[target]

    unit = "months" if run_config.freq in ['ME', 'M'] else "days"
    print(f"Engineered {len(features)} features.")
    print(f"Training on {len(X_train)} {unit}, testing on {len(X_test)} {unit}.")

    model = xgb.XGBRegressor(
        n_estimators=run_config.n_estimators,
        learning_rate=run_config.learning_rate,
        max_depth=run_config.max_depth,
        subsample=run_config.subsample,
        colsample_bytree=run_config.colsample_bytree,
        objective='reg:squarederror',
        random_state=42,
        early_stopping_rounds=run_config.early_stopping_rounds
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_train, y_train), (X_test, y_test)],
        verbose=False  # Set to False for cleaner console like A/B models
    )

    # Predict Test Set
    test_preds = model.predict(X_test)

    # Populate Metrics (matching config structure)
    metrics = ForecastingMetrics(run=f"{run_config.run}_metrics")
    metrics.mae = float(mean_absolute_error(y_test, test_preds))
    metrics.rmse = float(np.sqrt(mean_squared_error(y_test, test_preds)))
    metrics.elapsed_time_min = (time.time() - start_time) / 60

    # Autoregressive Future Prediction
    future_dates = pd.date_range(
        start=df.index[-1],
        periods=run_config.future_steps + 1,
        freq=run_config.freq
    )[1:]

    future_preds = []
    # Keep a running history buffer for recursive feature generation
    history_demand = list(df['demand'].values)

    for i in range(run_config.future_steps):
        curr_date = future_dates[i]
        feat_dict = {}

        # Calendar features
        feat_dict['dayofweek'] = curr_date.dayofweek
        feat_dict['month'] = curr_date.month
        feat_dict['dayofmonth'] = curr_date.day
        feat_dict['is_weekend'] = int(curr_date.dayofweek in [5, 6])

        # Lags
        for lag in lags:
            feat_dict[f'lag_{lag}'] = history_demand[-lag]

        # Rolling stats
        feat_dict[f'rolling_mean_{window_short}'] = np.mean(
            history_demand[-window_short:])
        feat_dict[f'rolling_std_{window_short}'] = np.std(
            history_demand[-window_short:], ddof=1) if len(history_demand) > 1 else 0
        feat_dict[f'rolling_mean_{window_long}'] = np.mean(
            history_demand[-window_long:])

        # Enforce exact column order
        curr_X = pd.DataFrame([feat_dict])[features]

        # Predict and update buffer
        pred = model.predict(curr_X)[0]
        future_preds.append(float(pred))
        history_demand.append(float(pred))

    print(f"\n--- Final Results ---")
    print(f"Test MAE:  {metrics.mae:.4f}")
    print(f"Test RMSE: {metrics.rmse:.4f}")

    # Use run_utils to save JSON and generate exact same HTML Plotly dashboards
    run_utils.save_run_data(run_config, metrics, output_dir)
    run_utils.plot_forecasting_dashboard(
        run_config=run_config,
        dates=df.index,
        data=df['demand'].values,
        test_dates=test_df.index,
        test_preds=test_preds,
        future_dates=future_dates,
        future_preds=future_preds,
        metrics=metrics,
        output_dir=output_dir
    )

    # Export XGBoost Model safely
    model.save_model(os.path.join(output_dir, f'{run_config.run}_model.json'))


if __name__ == "__main__":
    run_xgboost_forecaster("C1")
    run_xgboost_forecaster("C2")
    run_xgboost_forecaster("C3")
    run_xgboost_forecaster("C4")


## forecasting_experiments.py
Time-series forecasting pipeline comparing the SARIMAX and LSTM models in their ability to predict grocery demand.
### LSTM
LSTM (Long Short-Term Memory) is a deep learning model designed to detect and understand long-range dependencies in sequential data. As opposed to other neural networks, LSTMs use cell states and gating mechanisms to decide on which historical data to retain or forget.
- A `lookback` window is defined to predict the next value in the series.
- With the use of the `ReLU` activation function and multiple hidden layers, the LSTM can identify complex and non-linear patterns (e.g. a surge in demand that only occurs during weekends)
### SARIMAX
SARIMAX (Seasonal Autoregressive Integrated Moving Average with Exogenous Regressors) is a statistical model used for decomposing the time series into understandable mathematical components. It is trained directly on the chronological residuals of the series.
- Uses the relationship between an observation and a number of lagged observation
- Differentiates the data to make it stationary (removing long-term trends)
- Models forecast error as a linear combination of the error terms from previous steps
- Captures repeating cycles (e.g. 7 day weekly patterns or 12 month annual cycles)

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.statespace.sarimax import SARIMAX
from safetensors.torch import save_file

from config import get_run_config, ForecastingMetrics
import run_utils


def prepare_true_timeseries(filepath='groceries_dataset.csv', freq='D'):
    """
    Builds a TRUE chronological time-series from raw timestamp data.
    By default, calculates total daily grocery demand. If freq='W', 
    calculates total weekly grocery demand to reduce noise.
    """
    df = pd.read_csv(filepath)

    # Convert to datetime and sort chronologically
    df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y')
    df = df.sort_values(by='Date')

    # Handle aggregation based on frequency config
    if freq in ['W', 'ME', 'M']:
        aggregated_demand = df.groupby(pd.Grouper(
            key='Date', freq=freq)).size().reset_index(name='demand')
    else:
        # Default Daily aggregation
        aggregated_demand = df.groupby(
            'Date').size().reset_index(name='demand')

    aggregated_demand.set_index('Date', inplace=True)

    # Reindex missing dates/weeks with zero demand to maintain strict chronology
    full_date_range = pd.date_range(
        start=aggregated_demand.index.min(), end=aggregated_demand.index.max(), freq=freq)
    aggregated_demand = aggregated_demand.reindex(
        full_date_range, fill_value=0)

    return aggregated_demand['demand'].values.astype(float), aggregated_demand.index


def create_lstm_sequences(data, lookback):
    X, y = [], []
    for i in range(len(data) - lookback):
        X.append(data[i:(i + lookback)])
        y.append(data[i + lookback])
    return np.array(X), np.array(y)


class TimeSeriesLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=1):
        super(TimeSeriesLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size,
                            num_layers=num_layers, batch_first=True)
        self.fc1 = nn.Linear(hidden_size, 32)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(32, 1)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        last_out = lstm_out[:, -1, :]
        out = self.fc1(last_out)
        out = self.relu(out)
        out = self.fc2(out)
        return out


def run_sarimax(run_id="B1", output_dir="forecasting_results"):
    run_config = get_run_config(run_id)
    if not run_config:
        return

    print(f"\n--- Starting SARIMAX Run: {run_config.run} ---")
    start_time = time.time()

    # Reason: monthly ACF/PACF is cleaner than weekly -> better forecasting signal
    data, dates = prepare_true_timeseries(
        freq=getattr(run_config, 'freq', 'D'))
    train_size = int(len(data) * 0.8)
    train_data, test_data = data[:train_size], data[train_size:]

    model = SARIMAX(train_data, order=run_config.order, seasonal_order=run_config.seasonal_order,
                    enforce_stationarity=False, enforce_invertibility=False)
    # Increase maxiter to help B2 converge on noisy daily data
    fitted_model = model.fit(disp=False, maxiter=200)

    os.makedirs(output_dir, exist_ok=True)
    # NOTE: SARIMAX relies on statsmodels objects; cannot use .safetensors
    fitted_model.save(os.path.join(output_dir, f'{run_config.run}.pkl'))

    full_forecast = fitted_model.forecast(
        steps=len(test_data) + run_config.future_steps)
    test_preds = full_forecast[:len(test_data)]
    future_preds = full_forecast[len(test_data):]

    metrics = ForecastingMetrics(run=f"{run_config.run}_metrics")
    metrics.mae = mean_absolute_error(test_data, test_preds)
    metrics.rmse = np.sqrt(mean_squared_error(test_data, test_preds))
    metrics.elapsed_time_min = (time.time() - start_time) / 60

    run_utils.save_run_data(run_config, metrics, output_dir)

    test_dates = dates[train_size:]
    # Update future_dates to use the appropriate frequency (monthly, weekly, daily)
    future_dates = pd.date_range(
        start=dates[-1], periods=run_config.future_steps + 1, freq=getattr(run_config, 'freq', 'D'))[1:]
    run_utils.plot_forecasting_dashboard(
        run_config, dates, data, test_dates, test_preds, future_dates, future_preds, metrics, output_dir)


def run_forecasting_lstm(run_id="A1", output_dir="forecasting_results"):
    run_config = get_run_config(run_id)
    if not run_config:
        return

    print(f"\n--- Starting Forecasting LSTM Run: {run_config.run} ---")
    start_time = time.time()

    # Reason: monthly ACF/PACF is cleaner than weekly -> better forecasting signal
    data, dates = prepare_true_timeseries(
        freq=getattr(run_config, 'freq', 'D'))
    train_size = int(len(data) * 0.8)
    train_data, test_data = data[:train_size], data[train_size:]

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    scaler = MinMaxScaler()
    train_scaled = scaler.fit_transform(train_data.reshape(-1, 1)).flatten()
    test_scaled = scaler.transform(test_data.reshape(-1, 1)).flatten()

    X_train, y_train = create_lstm_sequences(
        train_scaled, run_config.lookback_window)
    X_test, y_test = create_lstm_sequences(
        test_scaled, run_config.lookback_window)

    if len(X_train) == 0 or len(X_test) == 0:
        print(f"Error: Not enough data for lookback_window={run_config.lookback_window} at freq={getattr(run_config, 'freq', 'D')}")
        return

    X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
    X_test = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

    train_loader = DataLoader(TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(
        y_train, dtype=torch.float32).unsqueeze(1)), batch_size=run_config.batch_size, shuffle=False)
    test_loader = DataLoader(TensorDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(
        y_test, dtype=torch.float32).unsqueeze(1)), batch_size=run_config.batch_size, shuffle=False)

    model = TimeSeriesLSTM(hidden_size=run_config.hidden_size,
                           num_layers=run_config.num_layers).to(device)
    optimizer = optim.Adam(model.parameters(), lr=run_config.learning_rate)

    loss_type = getattr(run_config, 'loss_type', 'MSE')
    if loss_type == "Huber":
        criterion = nn.HuberLoss()
    elif loss_type == "MAE":
        criterion = nn.L1Loss()
    else:
        criterion = nn.MSELoss()

    metrics = ForecastingMetrics(run=f"{run_config.run}_metrics")

    for epoch in range(run_config.epochs):
        model.train()
        total_loss = 0
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        metrics.train_loss.append(total_loss / len(train_loader))

    os.makedirs(output_dir, exist_ok=True)
    save_file(model.state_dict(), os.path.join(
        output_dir, f'{run_config.run}.safetensors'))

    model.eval()
    predictions_scaled = []
    with torch.no_grad():
        for batch_X, _ in test_loader:
            predictions_scaled.extend(
                model(batch_X.to(device)).cpu().numpy().flatten())

    predictions = scaler.inverse_transform(
        np.array(predictions_scaled).reshape(-1, 1)).flatten()
    y_test_true = scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()

    metrics.mae = mean_absolute_error(y_test_true, predictions)
    metrics.rmse = np.sqrt(mean_squared_error(y_test_true, predictions))
    metrics.elapsed_time_min = (time.time() - start_time) / 60

    # Future Inference
    last_seq = test_scaled[-run_config.lookback_window:
                           ].reshape(1, run_config.lookback_window, 1)
    future_preds_scaled = []
    with torch.no_grad():
        curr_seq = torch.tensor(last_seq, dtype=torch.float32).to(device)
        for _ in range(run_config.future_steps):
            pred = model(curr_seq)
            future_preds_scaled.append(pred.item())
            curr_seq = torch.cat(
                [curr_seq[:, 1:, :], pred.unsqueeze(1)], dim=1)

    future_forecast = scaler.inverse_transform(
        np.array(future_preds_scaled).reshape(-1, 1)).flatten()

    run_utils.save_run_data(run_config, metrics, output_dir)

    test_dates = dates[train_size + run_config.lookback_window:]
    future_dates = pd.date_range(
        start=dates[-1], periods=run_config.future_steps + 1, freq=getattr(run_config, 'freq', 'D'))[1:]
    run_utils.plot_forecasting_dashboard(
        run_config, dates, data, test_dates, predictions, future_dates, future_forecast, metrics, output_dir)


if __name__ == '__main__':
    # 1-2 Daily Runs (Baseline)
    # 3-4 Monthly Runs (Optimized Seasonality)
    run_forecasting_lstm("A1")
    run_forecasting_lstm("A2")
    run_forecasting_lstm("A3")
    run_forecasting_lstm("A4")

    run_sarimax("B1")
    run_sarimax("B2")
    run_sarimax("B3")
    run_sarimax("B4")


![table](image.png)

### Run results can be found in [_all_runs.pdf](_all_runs.pdf)
### [results.md](results.md) explains the conclusions drawn from the models